# Notebook 06 - Reading digits and turning them into numbers

*Time: about 55-60 minutes*

## What you will learn
- Recognise the digits **0-9** with the same template idea as Notebook 05.
- **Save** the digit templates as picture files so you can look at them.
- Glue recognised digits into a real **integer**: pixels `4 2 7` become the number `427`.
- A **confidence table** that shows *where* the reader is unsure.
- Why this is **not real OCR** (fonts, tilt and handwriting break templates).

## Prerequisites
- Notebooks 04 and 05 (boxes, sorting, template matching).

### What this code does

We load the libraries. `os` is Python's tool for folders and files (we use it to create a folder and list files).

In [ ]:
import sys                                    # lets us change where Python looks for files
sys.path.append("..")                         # also look one folder up (utils.py lives there)
import os                                     # folders and files
import cv2                                    # OpenCV
import numpy as np                            # NumPy: blank canvases
from utils import load, show_many             # our helpers
print("Libraries loaded - ready to go!")      # a visible sign that everything worked

### What you should see / Why we care

`Libraries loaded - ready to go!`

### What this code does

Three pictures today. **`IMAGE_PATH` (digits 0-9 in a row) is the one to change.**

In [ ]:
IMAGE_PATH = "../assets/digits_single.png"        # <- change me
NUMBER_PATH = "../assets/digits_number.png"       # two numbers: 427 and 1058
REAL_FONT_PATH = "../assets/digits_real_font.png" # a different, real-world font
print("pictures chosen:", IMAGE_PATH)

### What you should see / Why we care

`pictures chosen: ../assets/digits_single.png`

### What this code does

`find_boxes` is the same recipe as Notebook 05: gray, blur, Otsu, flip to white-on-black, contours, area filter, sort left to right.

In [ ]:
def find_boxes(gray, use_median, min_area):       # the recipe from Notebook 05
    if use_median:                                # which blur?
        blur = cv2.medianBlur(gray, 5)
    else:
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, otsu = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    thresh = cv2.bitwise_not(otsu)                # white digits on black
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []                                    # keep only big-enough shapes
    for contour in contours:
        if cv2.contourArea(contour) >= min_area:
            boxes.append(cv2.boundingRect(contour))
    return thresh, sorted(boxes)                  # left to right
print("find_boxes() is ready")

### What you should see / Why we care

`find_boxes() is ready`

### What this code does

We load the single digits, run the recipe and cut each digit out. Then we show the thumbnails.

In [ ]:
gray = cv2.cvtColor(load(IMAGE_PATH), cv2.COLOR_BGR2GRAY)   # load as gray
thresh, boxes = find_boxes(gray, False, 10)                 # white digits on black
print("digits found:", len(boxes))                          # we drew 10
crops = []                                                  # one small picture per digit
titles = []
for index in range(len(boxes)):                             # box 0, 1, 2 ...
    x, y, w, h = boxes[index]
    crops.append(thresh[y:y + h, x:x + w])                  # cut it out
    titles.append("box " + str(index))
show_many(crops, titles, columns=10, size=(14, 2))

### What you should see / Why we care

`digits found: 10` and ten thumbnails: **0 1 2 3 4 5 6 7 8 9**, each white on black. They are different widths (the `1` is narrow) and that is fine: we squash them all to the same size.

## Digit templates

Same trick as letters: draw each digit with `cv2.putText`, squash it to 32 x 32. This time we also **save** each template as a picture file, so students can open the folder `assets/digit_templates/` and *see* what the computer compares with.

### What this code does

`tidy` squashes any crop to 32 x 32. `FONT` and `DIGITS` are our settings.

In [ ]:
FONT = cv2.FONT_HERSHEY_SIMPLEX                             # the font OpenCV can draw
DIGITS = "0123456789"                                       # the ten digits

def tidy(crop):                                             # a recipe named tidy
    return cv2.resize(crop, (32, 32), interpolation=cv2.INTER_AREA)  # squash to 32x32

print("tidy ready:", tidy(crops[1]).shape, "(the narrow digit 1 now fills 32 x 32)")

### What you should see / Why we care

`tidy ready: (32, 32)`

### What this code does

For each digit: draw it in white on a black canvas, threshold it, find its tight box, cut it out, `tidy` it, store it in the dictionary `templates` and **write it to disk** with `cv2.imwrite`.

In [ ]:
folder = "../assets/digit_templates"                        # where the files will go
os.makedirs(folder, exist_ok=True)                          # create the folder if missing
templates = {}                                              # labelled drawer: digit -> picture
for digit in DIGITS:                                        # "0", "1", ... "9"
    canvas = np.zeros((120, 120), dtype=np.uint8)           # black canvas
    cv2.putText(canvas, digit, (20, 90), FONT, 2.5, 255, 4, cv2.LINE_AA)  # draw it white
    _, canvas = cv2.threshold(canvas, 127, 255, cv2.THRESH_BINARY)  # pure black/white
    x, y, w, h = cv2.boundingRect(canvas)                   # tight box
    templates[digit] = tidy(canvas[y:y + h, x:x + w])       # cut out, squash, store
    cv2.imwrite(folder + "/digit_" + digit + ".png", templates[digit])  # save the file
print("saved files:", sorted(os.listdir(folder)))           # prove they exist

### What you should see / Why we care

`saved files: ['digit_0.png', 'digit_1.png', ... 'digit_9.png']`. Open the folder `assets/digit_templates/` in your file explorer to look at them.

### What this code does

We show the ten templates in a grid.

In [ ]:
template_pictures = []                                      # pictures in digit order
for digit in DIGITS:
    template_pictures.append(templates[digit])              # take them from the drawer
show_many(template_pictures, list(DIGITS), columns=10, size=(14, 2))

### What you should see / Why we care

The ten digits as 32 x 32 blocks. This is the computer's whole "memory" of what digits look like.

### What this code does

`recognize` compares a crop with all ten templates. This version returns **four** things: the best digit, its score, the **runner-up** digit and its score. The runner-up helps us see how *close* the decision was.

In [ ]:
def recognize(crop, templates):                             # the matcher
    small = tidy(crop)                                      # squash to 32x32
    best, best_score, second, second_score = "?", -2.0, "?", -2.0   # starting values
    for digit in templates:                                 # try every template
        score = cv2.matchTemplate(small, templates[digit], cv2.TM_CCOEFF_NORMED)[0][0]
        if score > best_score:                              # new champion
            second, second_score = best, best_score         # old champion is runner-up
            best, best_score = digit, score
        elif score > second_score:                          # new runner-up
            second, second_score = digit, score
    return best, float(best_score), second, float(second_score)
print("recognize() ready:", recognize(crops[4], templates)[:2])

### What you should see / Why we care

`recognize() ready: ('4', 0.96...)`: it read the fifth thumbnail (the digit 4).

### What this code does

We recognise every one of the ten isolated digits and print what it thought, with a confidence. Then we count how many are right (the right answer for box `i` is the digit `i`).

In [ ]:
correct = 0                                                 # how many we got right
for index in range(len(boxes)):                             # box 0 ... 9
    x, y, w, h = boxes[index]
    digit, score, second, second_score = recognize(thresh[y:y + h, x:x + w], templates)
    print("box", index, "-> digit", digit, " confidence", round(score, 2))
    if digit == str(index):                                 # str(3) is the text "3"
        correct = correct + 1
print("correct:", correct, "out of", len(boxes))

### What you should see / Why we care

Ten lines like `box 0 -> digit 0  confidence 0.98` and the final line `correct: 10 out of 10`. Confidence values are all above 0.9.

## From digits to a real number

Now the big step. We want the pixels `4`, `2`, `7` to become the **integer 427**, a number we can add and multiply. A computer cannot add *pictures of digits*, but it can add *integers*.

The trick has three parts:
1. Collect the recognised digits as **text characters** in a list: `["4", "2", "7"]`.
2. `"".join(list)` glues them together into one string: `"427"`.
3. `int("427")` turns the string into the integer `427`.

So: **`number = int("".join(predicted_digits))`**

### What this code does

We try the trick with a hand-made list first, so you can see each stage. `type(...)` tells us what kind of thing a value is.

In [ ]:
predicted_digits = ["4", "2", "7"]                          # what the reader might give us
text = "".join(predicted_digits)                            # glue them: the TEXT "427"
number = int("".join(predicted_digits))                     # turn the text into an INTEGER
print("text  :", text, "  type:", type(text).__name__)      # str = text
print("number:", number, "  type:", type(number).__name__)  # int = whole number
print("proof it is a number: number + 1 =", number + 1)     # you cannot do this with text

### What you should see / Why we care

`text: 427  type: str`, `number: 427  type: int` and `number + 1 = 428`. The text `"427"` and the integer `427` print the same, but only the integer can be used in maths.

### What this code does

Now with the real picture. `digits_number.png` contains two numbers on one line. We find the boxes, recognise every digit, and watch the **gap** between boxes: when the gap is bigger than 80% of the digit's height, a **new number** starts. Each finished number is glued with `join` and turned into an integer.

In [ ]:
gray = cv2.cvtColor(load(NUMBER_PATH), cv2.COLOR_BGR2GRAY)  # the picture with two numbers
thresh, boxes = find_boxes(gray, False, 10)                 # white digits on black
numbers, current, last_right = [], [], None                 # finished numbers / digits so far
for x, y, w, h in boxes:                                    # left to right
    digit, score, second, second_score = recognize(thresh[y:y + h, x:x + w], templates)
    if last_right is not None and x - last_right > 0.8 * h: # a big gap: a new number starts
        numbers.append(int("".join(current)))               # finish the old number
        current = []                                        # start collecting again
    current.append(digit)                                   # add this digit
    last_right = x + w
numbers.append(int("".join(current)))                       # finish the last number
print("digits found:", len(boxes), "| numbers found:", numbers)

### What you should see / Why we care

`digits found: 7 | numbers found: [427, 1058]`. The pixels were turned into the integers **427** and **1058**.

### What this code does

Real integers can be used in maths. We add them and print the types.

In [ ]:
total = numbers[0] + numbers[1]                             # 427 + 1058
print(numbers[0], "+", numbers[1], "=", total)              # the sum
print("is it a real integer?", type(total).__name__)        # int
print("numbers read from the picture, doubled:", numbers[0] * 2, numbers[1] * 2)

### What you should see / Why we care

`427 + 1058 = 1485`, type `int`, and the doubled values `854 2116`. Reading pixels gave us real numbers.

## Where does it fail? A confidence table

A good reader does not just answer; it also shows **how sure** it is. We print one line per digit: the answer, its confidence, the runner-up and the **margin** (difference between best and runner-up). A small margin means "it was nearly a different digit".

### What this code does

The `"{:>6.2f}"` pieces are formatting: `>6` means "right-align in 6 characters" and `.2f` means "2 decimals". Rows with a confidence below 0.9 get a warning flag.

In [ ]:
print("pos    x  digit    conf  runner-up    conf  margin")      # column titles
for index in range(len(boxes)):                                  # every box
    x, y, w, h = boxes[index]
    digit, score, second, second_score = recognize(thresh[y:y + h, x:x + w], templates)
    flag = ""                                                    # no warning by default
    if score < 0.9:                                              # not confident?
        flag = "  <-- check!"
    print("{:>3} {:>4}  {:>5} {:>7.2f}  {:>9} {:>7.2f} {:>7.2f}{}".format(
        index, x, digit, score, second, second_score, score - second_score, flag))

### What you should see / Why we care

A table with 7 rows. The answer for each row is right (`4 2 7 1 0 5 8`), the confidence is above 0.9 and the margins are large (0.3 or more). On clean, same-font digits the reader is safe. The next section shows what happens when we leave this comfort zone.

## Why this is NOT real OCR

Template matching works here because **the pictures use exactly the same font, the same orientation, and little noise as our templates**. Real OCR (Optical Character Recognition) must cope with:
- **fonts** that look different (a "1" with a flag and base, a slashed zero, a curly 7),
- **tilt**, **blur**, **noise**, **shadows**,
- **handwriting**, where no two "7"s look alike.

You cannot make a template for every possible handwriting, so real OCR *learns* what a "7" looks like from **millions of examples**: that is machine learning. Notebook 09 shows this.

### What this code does

**Test 1: tilt.** We rotate `digits_number.png` by different angles and read all 7 digits again (as a plain string of digits, no grouping).

In [ ]:
height, width = gray.shape                                       # picture size
for angle in [0, 8, 12, 15, 20]:                                 # degrees of tilt
    matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, 1.0)
    tilted = cv2.warpAffine(gray, matrix, (width, height), borderValue=255)
    t, b = find_boxes(tilted, False, 10)                         # same recipe
    answer = ""                                                  # the digits read so far
    for x, y, w, h in b:
        answer = answer + recognize(t[y:y + h, x:x + w], templates)[0]
    print("tilt", str(angle).rjust(2), "deg ->", answer, "(correct is 4271058)")

### What you should see / Why we care

`4271058` stays right up to around 8 degrees, then errors creep in (`4291058` at 12 degrees, `6191058` at 15 degrees). A human can read tilted digits without any effort. Our templates cannot.

### What this code does

**Test 2: a different font.** `digits_real_font.png` was drawn with a normal computer font (it has a dotted zero and different shapes). We cut out just its first number, which really says **2026**, and read it with the same code.

In [ ]:
real = cv2.cvtColor(load(REAL_FONT_PATH), cv2.COLOR_BGR2GRAY)    # the real-font picture
region = real[80:180, 40:260]                                    # only the first number
t, b = find_boxes(region, False, 10)                             # same recipe
answer = ""                                                      # digits read so far
for x, y, w, h in b:
    answer = answer + recognize(t[y:y + h, x:x + w], templates)[0]
print("the picture says 2026, we read:", answer)
show_many([region], ["real-font digits: 2026"], columns=1, size=(5, 3))

### What you should see / Why we care

`the picture says 2026, we read: 2025`. The last digit (a 6 in a new font) was mistaken for a 5, and nobody told the program it was wrong. Same code, different font, wrong answer. **That** is the problem machine learning solves.

## Recap
- **digits -> templates -> best match** = a digit reader.
- `int("".join(predicted_digits))` turns recognised digits into a real number.
- The **confidence table** shows where the reader is weak.
- It fails on **tilt and unfamiliar fonts**, and that is why real OCR uses trained models.

**Next:** Notebook 07 finds and cuts out **objects** (not just characters).